# MNIST MLP3 Muon: rectangular RG spectra for FC1 and FC2

This notebook replaces the pseudoinverse-only flow for rectangular matrices with a gauge-aligned decomposition.

For a wide full-row-rank matrix such as

\[
W_t\in\mathbb{R}^{512\times 784},
\]

write

\[
W_t=B_tV_t^\top,
\qquad V_t^\top V_t=I.
\]

The row-space bases at successive steps are aligned by orthogonal Procrustes. The square core flow is then

\[
J_t^{\mathrm{core}}=\widetilde B_t B_{t-1}^{-1},
\]

with spectrum

\[
x_i^{\mathrm{core}}=\left|\log\sigma_i^2(J_t^{\mathrm{core}})\right|.
\]

The quotient/angular sector is measured independently by the principal angles between the two row spaces:

\[
x_i^{\mathrm{angular}}=\theta_i^2.
\]

For square full-rank `fc2.weight`, the angular sector vanishes and the core operator reduces to

\[
J_t^{\mathrm{core}}=W_tW_{t-1}^{-1}.
\]

Both spectra are fitted with `powerlaw.Fit`, using the explicit exponent range \(1.01\leq\alpha\leq10\).

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from rg_baselines.mnist_muon_rectangular_analysis import (
    analyze_rectangular_muon_run,
)


In [ ]:
RUN_DIR = Path("../results/mnist_mlp3_muon_microbatch")
OUTPUT_DIR = RUN_DIR / "rectangular_rg_analysis"

# Use 1 for every successive microbatch. Larger values reduce fitting cost,
# while each selected step is still compared with the immediately prior step.
STEP_STRIDE = 1
MAX_STEP = None
POWERLAW_ALPHA_RANGE = (1.01, 10.0)
ANGLE_ZERO_TOL = 1e-12


In [ ]:
RESULT = analyze_rectangular_muon_run(
    RUN_DIR,
    output_dir=OUTPUT_DIR,
    layers=("fc1.weight", "fc2.weight"),
    step_stride=STEP_STRIDE,
    max_step=MAX_STEP,
    angle_zero_tol=ANGLE_ZERO_TOL,
    powerlaw_alpha_range=POWERLAW_ALPHA_RANGE,
)

RESULT["summary"]


In [ ]:
display(RESULT["summary"])
display(RESULT["diagnostics"].groupby("layer").tail(5))


In [ ]:
for filename in (
    "alpha_core_log_deviation_vs_step.png",
    "alpha_angular_theta_squared_vs_step.png",
    "maximum_principal_angle_vs_step.png",
    "esd_core_log_deviation_fc1_weight.png",
    "esd_angular_theta_squared_fc1_weight.png",
    "esd_core_log_deviation_fc2_weight.png",
):
    path = OUTPUT_DIR / filename
    print(path)
    display(Image(filename=str(path)))
